The [navigation MDP section](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index) wrote TurtleBot3 navigation as an MDP with 28 continuous state variables and a continuous action. It then solved a finite version by value iteration, which needed the robot's true pose, a fixed goal and a table of transition probabilities built from the simulator. Q-learning needs no such table: it learns from the transitions the robot experiences. [Q-learning](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/control/q-learning/index) in its tabular form needs a finite set of states and a finite set of actions, so that $q(s, a)$ fits in a table. In this section you make the problem finite, train a tabular agent on it, and see where the coarse state stops being good enough.

The simulator, reward and episode rules are the ones from that section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

from cycler import cycler
from matplotlib_inline.backend_inline import (
    set_matplotlib_formats,
)

set_matplotlib_formats("svg")
FIG_GRAY = "#808080"
FIG_COLORS = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#cb77a6",
    "#c28600",
    "#1d99e0",
    "#7f7f7f",
]
plt.rcParams.update(
    {
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "savefig.facecolor": "none",
        "savefig.transparent": True,
        "text.color": FIG_GRAY,
        "axes.labelcolor": FIG_GRAY,
        "axes.edgecolor": FIG_GRAY,
        "xtick.color": FIG_GRAY,
        "ytick.color": FIG_GRAY,
        "grid.color": FIG_GRAY,
        "grid.alpha": 0.25,
        "legend.facecolor": "none",
        "legend.edgecolor": FIG_GRAY,
        "legend.framealpha": 0,
        "axes.prop_cycle": cycler(color=FIG_COLORS),
        "svg.hashsalt": "aiml",
        "svg.fonttype": "path",
    }
)
BLUE, ORANGE, GREEN, PINK, GOLD = FIG_COLORS[:5]

In [ ]:
V_MAX, W_MAX = 0.26, 1.82  # m/s, rad/s (Waffle limits)
RADIUS = 0.2  # m, robot footprint
DT = 0.2  # s, control period
N_BEAMS = 24
R_MIN, R_MAX = 0.12, 3.5  # m, lidar span
ARENA = 4.0  # m, side of the square room
GOAL_TOL = 0.25  # m
T_MAX = 200  # steps per episode (40 s)

# Obstacles: (x, y, radius) in meters
OBSTACLES = np.array(
    [
        [1.2, 1.3, 0.25],
        [2.8, 1.1, 0.20],
        [2.0, 2.4, 0.30],
        [3.0, 3.0, 0.20],
    ]
)


def ray_cast(x, y, angles):
    """Lidar range along each world-frame angle."""
    dx, dy = np.cos(angles), np.sin(angles)
    r = np.full(angles.shape, np.inf)
    with np.errstate(divide="ignore", invalid="ignore"):
        for wall, d, p in (
            (0.0, dx, x),
            (ARENA, dx, x),
            (0.0, dy, y),
            (ARENA, dy, y),
        ):
            t = (wall - p) / d
            r = np.minimum(r, np.where(t > 0, t, np.inf))
    for cx, cy, rho in OBSTACLES:
        ox, oy = x - cx, y - cy
        b = ox * dx + oy * dy
        disc = b * b - (ox * ox + oy * oy - rho * rho)
        t = -b - np.sqrt(np.maximum(disc, 0.0))
        r = np.minimum(
            r, np.where((disc >= 0) & (t > 0), t, np.inf)
        )
    return np.clip(r, R_MIN, R_MAX)


def clearance(x, y):
    """Distance from a point to the nearest surface."""
    walls = min(x, ARENA - x, y, ARENA - y)
    centers = np.hypot(OBSTACLES[:, 0] - x, OBSTACLES[:, 1] - y)
    return min(walls, np.min(centers - OBSTACLES[:, 2]))


class NavEnv:
    """TurtleBot3 Waffle driving to a goal in a 4 m room."""

    def __init__(self, seed=0, noise=0.02, collision_cost=20.0):
        self.collision_cost = collision_cost
        # Starts and goals. Each reset also seeds that episode's noise,
        # so episode k is the same whatever the earlier episodes did.
        self.rng = np.random.default_rng(seed)
        self.noise = noise
        self.beam_angles = np.linspace(
            -np.pi, np.pi, N_BEAMS, endpoint=False
        )

    def _free_point(self, margin):
        while True:
            p = self.rng.uniform(margin, ARENA - margin, size=2)
            if clearance(*p) > margin:
                return p

    def reset(self, start=None, goal=None):
        self.noise_rng = np.random.default_rng(self.rng.integers(2**63))
        if goal is None:
            goal = self._free_point(0.4)
        self.goal = np.array(goal, float)
        if start is None:
            while True:
                p = self._free_point(0.4)
                if np.hypot(*(p - self.goal)) > 1.5:
                    break
            start = (*p, self.rng.uniform(-np.pi, np.pi))
        self.pose = np.array(start, float)
        self.prev = np.zeros(2)
        self.t = 0
        self.dist = self.goal_distance()
        return self.observe()

    def goal_distance(self):
        return float(np.hypot(*(self.goal - self.pose[:2])))

    def goal_bearing(self):
        x, y, phi = self.pose
        g = np.arctan2(self.goal[1] - y, self.goal[0] - x) - phi
        return float(np.arctan2(np.sin(g), np.cos(g)))

    def scan(self):
        x, y, phi = self.pose
        return ray_cast(x, y, phi + self.beam_angles)

    def observe(self):
        """The 28 numbers the agent receives."""
        return np.concatenate(
            [
                self.scan() / R_MAX,
                [
                    min(self.dist, R_MAX) / R_MAX,
                    self.goal_bearing() / np.pi,
                    self.prev[0] / V_MAX,
                    self.prev[1] / W_MAX,
                ],
            ]
        ).astype(np.float32)

    def step(self, v, w):
        v = float(np.clip(v, 0.0, V_MAX))
        w = float(np.clip(w, -W_MAX, W_MAX))
        vn = v + self.noise * self.noise_rng.standard_normal()
        wn = w + self.noise * self.noise_rng.standard_normal()
        x, y, phi = self.pose
        x += vn * np.cos(phi) * DT
        y += vn * np.sin(phi) * DT
        phi = np.arctan2(
            np.sin(phi + wn * DT), np.cos(phi + wn * DT)
        )
        self.pose = np.array([x, y, phi])
        self.prev = np.array([v, w])
        self.t += 1
        d = self.goal_distance()
        progress, self.dist = self.dist - d, d
        r = 2.0 * progress - 0.05 * abs(w) / W_MAX - 0.01
        event, terminated = None, False
        if d < GOAL_TOL:
            r, event, terminated = r + 20.0, "goal", True
        elif clearance(x, y) < RADIUS:
            r -= self.collision_cost
            event, terminated = "collision", True
        truncated = not terminated and self.t >= T_MAX
        if truncated:
            event = "timeout"
        info = {"event": event}
        return self.observe(), r, terminated, truncated, info

In [ ]:
def draw_room(ax, env=None):
    ax.add_patch(
        Rectangle(
            (0, 0), ARENA, ARENA, fill=False, ec=FIG_GRAY, lw=1.5
        )
    )
    for cx, cy, rho in OBSTACLES:
        ax.add_patch(
            Circle((cx, cy), rho, color=FIG_GRAY, alpha=0.5, lw=0)
        )
    if env is not None:
        ax.plot(*env.goal, marker="*", ms=16, color=GOLD, ls="")
    ax.set_xlim(-0.1, ARENA + 0.1)
    ax.set_ylim(-0.1, ARENA + 0.1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")


def draw_robot(ax, pose, color=BLUE):
    x, y, phi = pose
    ax.add_patch(
        Circle((x, y), RADIUS, fill=False, ec=color, lw=1.5)
    )
    ax.plot(
        [x, x + RADIUS * np.cos(phi)],
        [y, y + RADIUS * np.sin(phi)],
        color=color,
        lw=1.5,
    )


def go_to_goal(env):
    b = env.goal_bearing()
    v = V_MAX * max(0.0, np.cos(b))
    return v, float(np.clip(2.0 * b, -W_MAX, W_MAX))


def rollout(env, policy, start=None, goal=None):
    env.reset(start=start, goal=goal)
    poses, rewards = [env.pose.copy()], []
    while True:
        _, r, term, trunc, info = env.step(*policy(env))
        poses.append(env.pose.copy())
        rewards.append(r)
        if term or trunc:
            return np.array(poses), np.array(rewards), info


def success_rate(policy, n=300, seed=99):
    env = NavEnv(seed=seed)
    events = {"goal": 0, "collision": 0, "timeout": 0}
    for _ in range(n):
        *_, info = rollout(env, policy)
        events[info["event"]] += 1
    return {k: v / n for k, v in events.items()}

## A finite version of the problem

The action set keeps five commands. Each one turns at a fixed rate, and the sharper turns drive more slowly:

| Action | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| $\omega$ (rad/s) | $-1.5$ | $-0.75$ | 0 | 0.75 | 1.5 |
| $v$ (m/s) | 0.10 | 0.18 | 0.26 | 0.18 | 0.10 |

The state keeps two pieces of the 28 numbers, each cut into a few bins:

- The goal bearing $\beta_t$ falls into one of 8 sectors of 45 degrees, centered on straight ahead, left, behind, right and the four diagonals.
- The lidar is reduced to three sectors: front ($\pm 45$ degrees), left and right (45 to 135 degrees to each side). Each sector reports its shortest range as near (under 0.45 m), middle (0.45 to 0.9 m) or far.

That gives $8 \times 3^3 = 216$ states and a table of $216 \times 5 = 1080$ action values. The goal distance, the beams behind the robot and the previous command are dropped.

In [ ]:
W_ACT = np.array([-1.5, -0.75, 0.0, 0.75, 1.5])
V_ACT = np.array([0.10, 0.18, 0.26, 0.18, 0.10])
N_STATES, N_ACTIONS = 216, 5
NEAR, MID = 0.45, 0.9  # m, range bin edges


def sector_bin(ranges, angles, lo, hi):
    m = ranges[(angles >= lo) & (angles < hi)].min()
    return 0 if m < NEAR else (1 if m < MID else 2)


def discretize(env):
    """Map the robot's situation to one of 216 states."""
    ranges, ang = env.scan(), env.beam_angles
    front = sector_bin(ranges, ang, -np.pi / 4, np.pi / 4)
    left = sector_bin(ranges, ang, np.pi / 4, 3 * np.pi / 4)
    right = sector_bin(ranges, ang, -3 * np.pi / 4, -np.pi / 4)
    b = env.goal_bearing()
    bearing = int(((b + np.pi) / (2 * np.pi) * 8 + 0.5) % 8)
    return ((bearing * 3 + front) * 3 + left) * 3 + right

In [ ]:
from matplotlib.patches import Wedge

env = NavEnv(seed=3)
env.reset(start=(1.75, 1.35, 0.9), goal=(3.5, 3.5))
fig, ax = plt.subplots(figsize=(5.4, 5.4))
draw_room(ax, env)
x, y, phi = env.pose
sectors = [
    ("front", -45, 45, BLUE),
    ("left", 45, 135, GREEN),
    ("right", -135, -45, PINK),
]
for name, lo, hi, c in sectors:
    for r_out, alpha in ((MID, 0.12), (NEAR, 0.25)):
        ax.add_patch(
            Wedge(
                (x, y),
                r_out,
                np.degrees(phi) + lo,
                np.degrees(phi) + hi,
                color=c,
                alpha=alpha,
                lw=0,
            )
        )
    mid = phi + np.radians((lo + hi) / 2)
    ax.text(
        x + 1.05 * np.cos(mid),
        y + 1.05 * np.sin(mid),
        name,
        color=c,
        ha="center",
        va="center",
    )
draw_robot(ax, env.pose)
ranges = env.scan()
for a, r in zip(phi + env.beam_angles, ranges):
    ax.plot(
        [x, x + r * np.cos(a)],
        [y, y + r * np.sin(a)],
        color=ORANGE,
        lw=0.6,
        alpha=0.5,
    )
s = discretize(env)
ax.set_title(f"Discrete state {s}")
ax.set_xlim(0.4, 3.9)
ax.set_ylim(0.4, 3.9)
plt.tight_layout()
plt.show()

*The three lidar sectors around the robot. The darker inner band is "near" (under 0.45 m) and the lighter band is "middle" (under 0.9 m); beyond it a sector reads "far". Only the shortest beam in each sector counts, and the beams behind the robot are ignored.*

## The Q-learning update

Q-learning keeps a table $Q(s, a)$ and moves one entry after every step toward a target built from the reward and the best action value in the next state:

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \big].
$$

This is the update of [TD prediction](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/prediction/temporal-difference) written for action values. The term in brackets is the TD error,

$$
\delta_t = R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t),
$$

the gap between the TD target and the current estimate. The update moves $Q(S_t, A_t)$ a fraction $\alpha$ of the way across that gap. The [Monte Carlo](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/prediction/monte-carlo) estimate of the navigation MDP section did something else: it waited until an episode ended and averaged the actual returns $G_t$. TD bootstraps instead. It updates after every 0.2 s step, using the table's own estimate of the next state in place of the rest of the return. An episode here lasts up to 200 steps, so TD makes up to 200 updates in the time Monte Carlo waits for one return. The [comparison of TD and Monte Carlo](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/prediction/td-vs-mc-mrp/td-vs-mc-mrp) shows the trade: the TD target has lower variance, and it is biased while the table is still wrong.

Compare the target with the value-iteration backup of the [navigation MDP section](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index), written for action values:

$$
q_{k+1}(s, a) = \mathcal R^a_s + \gamma \sum_{s'} \mathcal P^a_{ss'} \max_{a'} q_k(s', a').
$$

Value iteration takes the expected reward and the sum over next states from the model. Q-learning replaces both with the one reward and the one next state that the robot actually experienced. One sample is a noisy estimate of the expectation. The step size $\alpha$ averages many of them, so over repeated visits $Q(s, a)$ moves toward the mean of the targets, which is the backup applied to the current table. That is why Q-learning needs no model: the robot's own transitions are samples from $\mathcal P^a_{ss'}$.

After a goal or a collision the next state has no future, so the target is $R_{t+1}$ alone. After a timeout the robot still has a future, so the update bootstraps as usual. The agent explores with an $\epsilon$-greedy policy: with probability $\epsilon$ it picks a random action, and otherwise the action with the largest $Q(S_t, a)$. Here $\epsilon$ falls linearly from 1 to 0.05 over the first 60% of training, $\alpha = 0.1$ and $\gamma = 0.99$.

The update uses the greedy action in the target while the robot follows the $\epsilon$-greedy one, which makes Q-learning off-policy. [SARSA](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/control/sarsa/index) would use the action the robot actually takes next:

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \big].
$$

SARSA learns the value of the $\epsilon$-greedy policy itself. That value includes the random actions, some of which turn the robot into an obstacle. Q-learning learns the value of the greedy policy, whatever the exploration does. The [Q-learning](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/control/q-learning/index) section shows the consequence on the cliff-walking gridworld: SARSA learns a safer path that stays away from the cliff, and Q-learning learns the shortest one. In a finite MDP, the table converges to $q_*$ if every state-action pair keeps being visited and the step size shrinks over time. Neither holds exactly here: $\alpha$ stays at 0.1, and the end of this section shows that the discrete state is not quite Markov. The $\epsilon$ floor of 0.05 keeps every action in use.

In [ ]:
def q_learning(
    n_episodes=4000, alpha=0.1, gamma=0.99, eps_min=0.05, seed=0
):
    env = NavEnv(seed=seed)
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    visits = np.zeros((N_STATES, N_ACTIONS), dtype=int)
    events, returns = [], []
    for ep in range(n_episodes):
        eps = max(eps_min, 1 - ep / (0.6 * n_episodes))
        env.reset()
        s, G, k = discretize(env), 0.0, 0
        while True:
            if rng.random() < eps:
                a = int(rng.integers(N_ACTIONS))
            else:
                a = int(np.argmax(Q[s]))
            _, r, term, trunc, info = env.step(V_ACT[a], W_ACT[a])
            s2 = discretize(env)
            target = r if term else r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            visits[s, a] += 1
            G += gamma**k * r
            s, k = s2, k + 1
            if term or trunc:
                events.append(info["event"])
                returns.append(G)
                break
    return Q, visits, events, np.array(returns)


Q, visits, events, returns = q_learning()

In [ ]:
def rolling(x, w=200):
    return np.convolve(x, np.ones(w) / w, mode="valid")


goal_hit = np.array([e == "goal" for e in events], float)
crash = np.array([e == "collision" for e in events], float)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ep = np.arange(len(rolling(goal_hit))) + 200
ax.plot(ep, rolling(goal_hit), color=GREEN, label="goal")
ax.plot(ep, rolling(crash), color=ORANGE, label="collision")
ax.axvline(0.6 * len(events), color=FIG_GRAY, ls=":", lw=1)
ax.set_xlabel("episode")
ax.set_ylabel("fraction of last 200 episodes")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Outcomes during training")
ax2.plot(ep, rolling(returns), color=BLUE)
ax2.axvline(0.6 * len(events), color=FIG_GRAY, ls=":", lw=1)
ax2.set_xlabel("episode")
ax2.set_ylabel("$G_0$, mean of last 200")
ax2.set_title("Discounted return during training")
plt.tight_layout()
plt.show()

*Outcomes and discounted returns while the agent trains, averaged over windows of 200 episodes. The dotted line marks where $\epsilon$ reaches its floor of 0.05.*

At the start the robot acts at random and collides in most episodes. As $\epsilon$ falls, the agent acts more on its table and the goal rate climbs, to about 0.8 when $\epsilon$ reaches its floor. It does not stay there. The goal rate falls back to about 0.7 for roughly 1,000 episodes, dips to 0.6 near episode 3,500, and ends near 0.85.

To compare with the go-to-goal controller from the previous section, run the greedy policy, with no exploration, on the same 300 test episodes:

In [ ]:
def greedy(env):
    a = int(np.argmax(Q[discretize(env)]))
    return V_ACT[a], W_ACT[a]


print("go-to-goal:", success_rate(go_to_goal))
print("Q-learning:", success_rate(greedy))

In [ ]:
starts = [
    (0.6, 3.4, 0.0),
    (3.6, 1.8, 1.6),
    (0.5, 0.5, 0.0),
    (2.0, 0.5, 1.6),
]
goal = (3.5, 3.5)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
for ax, (name, pol) in zip(
    axes, [("go-to-goal", go_to_goal), ("Q-learning", greedy)]
):
    draw_room(ax)
    ax.plot(*goal, marker="*", ms=16, color=GOLD, ls="")
    env = NavEnv(seed=0)
    for k, s in enumerate(starts):
        poses, _, info = rollout(env, pol, s, goal)
        ax.plot(
            poses[:, 0],
            poses[:, 1],
            color=FIG_COLORS[k],
            lw=1.5,
            label=info["event"],
        )
        draw_robot(ax, poses[-1], FIG_COLORS[k])
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title(name)
plt.tight_layout()
plt.show()

*The four starts of the navigation MDP section, driven by the go-to-goal controller (left) and by the greedy Q-learning policy (right).*

The learned policy uses the lidar. When the front sector reads near, it turns away and drives around the obstacle, then turns back toward the goal once the goal bearing changes bin. That is the behavior the go-to-goal controller lacked, and it raises the success rate well above it. It is still not perfect. From the bottom left start it drives around the large obstacle and then collides with the small one at (3.0, 3.0), short of the goal. The next part explains why.

The value-iteration policy of the navigation MDP section reached the goal in every test episode. The two results answer different problems. That policy was computed for one goal, from the true pose and a model of the dynamics. The Q-table works for any goal, from what the robot's own sensors measure, and it never saw a model. The price is the state: 216 coarse bins in place of 16,229 pose cells.

## Where the table falls short

Two things limit the tabular agent: how often it sees each state, and how much the state hides.

In [ ]:
counts = visits.sum(axis=1)
order = np.argsort(counts)[::-1]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(np.arange(N_STATES), counts[order], color=BLUE, width=1.0)
ax.set_yscale("log")
ax.set_xlabel("states, sorted by visit count")
ax.set_ylabel("visits during training")
ax.set_title("How often each of the 216 states was visited")
plt.tight_layout()
plt.show()
print(
    "states visited fewer than 50 times:",
    int((counts < 50).sum()),
    "of",
    N_STATES,
)

*Number of training steps spent in each discrete state, sorted, on a log scale.*

The visits are very uneven. The most common state, "goal ahead, middle range in all three sectors", took more than 10,000 steps, while states such as "goal behind, obstacles near on both sides" occurred a few times or never. A rarely visited state has an action value estimated from few samples, and the policy there is close to a guess. Finer bins would make this worse: every extra bin multiplies the number of states, and each state is visited less.

The larger problem is aliasing. Many physical situations share one discrete state, and they can need different actions. An obstacle at 0.5 m and one at 0.85 m both read "middle". A goal 5 degrees to the left and one 40 degrees to the left fall in the same bearing bin. A pole just behind the left sector boundary is invisible to it. The table stores one value per state and action, so it learns the average return over all the situations that look alike, and in each of them the chosen action is right only on average. In MDP terms, the discrete state is not Markov: the next discrete state depends on where inside the bin the robot really is, and the table cannot see that. The pose grid of the navigation MDP section was close to Markov because it kept the position to within 11 cm and the heading to within 22.5 degrees. This state keeps neither.

You cannot fix this with more bins. Ten range bins for each of the 24 beams would give $10^{24}$ states, and nearly all of them would never be visited. The way out is to stop using a table: approximate $q(s, a)$, or the policy itself, with a function of the full 28 numbers, so that nearby states share what is learned. Deep Q-networks do this for discrete actions. The [next section](/aiml-common/lectures/reinforcement-learning/robot-navigation/ddpg-navigation/index) goes one step further and also drops the five-command action set, learning continuous $(v, \omega)$ with DDPG.